# 03 A/B Test Analysis

Experimental-style analysis of the relationship between price-increase exposure and customer churn in the synthetic customer dataset.

In [1]:
from math import erfc, sqrt
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "raw" / "customers.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CUSTOMERS_PATH = PROJECT_ROOT / "data" / "raw" / "customers.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

## Method Note

This notebook treats the synthetic price-increase exposure flag as an A/B-style comparison for analysis practice:

- Control group: customers not exposed to a price increase
- Treatment group: customers exposed to a price increase

This is not a causal inference analysis. The comparison below does not prove that the price increase caused churn, because exposure may still be related to customer attributes such as region, acquisition channel, tenure, purchase frequency, or prior spending.

## Load Customer Data

In [2]:
customers = pd.read_csv(
    CUSTOMERS_PATH,
    parse_dates=["first_purchase_date", "last_transaction_date"],
)

def normalize_boolean(series):
    """Normalize CSV boolean values while preserving invalid values as missing."""
    if pd.api.types.is_bool_dtype(series):
        return series
    return series.astype(str).str.lower().map({"true": True, "false": False})

for column in ["price_increase_occurred", "churned"]:
    customers[column] = normalize_boolean(customers[column])

customers["ab_group"] = customers["price_increase_occurred"].map(
    {False: "Control: no price increase", True: "Treatment: price increase"}
)
customers["ab_group"] = pd.Categorical(
    customers["ab_group"],
    categories=["Control: no price increase", "Treatment: price increase"],
    ordered=True,
)

display(customers.head())
display(customers[["customer_id", "ab_group", "churned"]].isna().sum().to_frame("missing_values"))

,customer_id,first_purchase_date,customer_region,acquisition_channel,customer_tenure_days,purchase_frequency,prior_spending,transaction_count,total_spending,average_discount_percent,price_increase_occurred,churned,last_transaction_date,ab_group
0,C000001,2025-10-16,uk,direct,76,8.0000,103.9800,2,103.9800,0.0000,False,True,2025-11-08,Control: no price increase
1,C000002,2025-07-12,uk,influencer,172,4.2470,120.8300,2,120.8300,12.5000,False,False,2025-10-31,Control: no price increase
2,C000003,2025-07-12,north_america,influencer,172,6.3710,170.7700,3,170.7700,6.6700,False,False,2025-12-25,Control: no price increase
3,C000004,2024-08-08,rest_of_world,influencer,510,5.7290,491.9300,8,491.9300,8.7500,False,False,2025-12-13,Control: no price increase
4,C000005,2025-07-03,uk,affiliate,181,4.0360,104.9700,2,104.9700,0.0000,True,False,2025-07-29,Treatment: price increase


,missing_values
customer_id,0
ab_group,0
churned,0


## Sample Size and Churn Rate by Group

In [3]:
group_summary = (
    customers.groupby("ab_group", observed=False)
    .agg(
        sample_size=("customer_id", "count"),
        churned_customers=("churned", "sum"),
        churn_rate=("churned", "mean"),
        avg_purchase_frequency=("purchase_frequency", "mean"),
        avg_customer_tenure_days=("customer_tenure_days", "mean"),
        avg_prior_spending=("prior_spending", "mean"),
        avg_discount_percent=("average_discount_percent", "mean"),
    )
    .reset_index()
)

display(group_summary)
display(
    group_summary[["ab_group", "sample_size", "churn_rate"]]
    .style.format({"churn_rate": "{:.2%}"})
    .bar(subset=["sample_size"], color="#4C78A8")
    .bar(subset=["churn_rate"], color="#E15759")
)

,ab_group,sample_size,churned_customers,churn_rate,avg_purchase_frequency,avg_customer_tenure_days,avg_prior_spending,avg_discount_percent
0,Control: no price increase,8929,595,0.0666,3.9755,356.8134,195.1018,7.9967
1,Treatment: price increase,3571,315,0.0882,4.5889,435.6847,297.8796,8.5767


,ab_group,sample_size,churn_rate
0,Control: no price increase,8929,6.66%
1,Treatment: price increase,3571,8.82%


## Group Context Checks

These checks are not causal adjustments. They show whether exposed and unexposed customers look different on observable customer characteristics.

In [4]:
region_balance = pd.crosstab(
    customers["customer_region"],
    customers["ab_group"],
    normalize="columns",
).round(4)

channel_balance = pd.crosstab(
    customers["acquisition_channel"],
    customers["ab_group"],
    normalize="columns",
).round(4)

display(region_balance)
display(channel_balance)

ab_group,Control: no price increase,Treatment: price increase
customer_region,,
asia_pacific,0.0974,0.0975
europe,0.1855,0.1918
north_america,0.3052,0.3022
rest_of_world,0.0675,0.0652
uk,0.3444,0.3433


ab_group,Control: no price increase,Treatment: price increase
acquisition_channel,,
affiliate,0.0760,0.0778
direct,0.1631,0.1649
email,0.1068,0.0963
influencer,0.1939,0.2016
organic_search,0.1707,0.1848
paid_social,0.2895,0.2744


## Treatment Effect Estimate and Statistical Test

The treatment effect is measured as the treatment churn rate minus the control churn rate. The confidence interval uses an unpooled standard error for the difference in proportions. The hypothesis test uses a two-sided, pooled two-proportion z-test.

In [5]:
def two_sided_normal_p_value(z_statistic):
    """Return the two-sided p-value for a standard normal z-statistic."""
    return erfc(abs(z_statistic) / sqrt(2))


def difference_in_proportions_ci(treatment_rate, control_rate, treatment_n, control_n, z_critical=1.959963984540054):
    """Calculate a 95% confidence interval for the difference in two proportions."""
    standard_error = sqrt(
        treatment_rate * (1 - treatment_rate) / treatment_n
        + control_rate * (1 - control_rate) / control_n
    )
    difference = treatment_rate - control_rate
    return difference - z_critical * standard_error, difference + z_critical * standard_error, standard_error


def two_proportion_z_test(treatment_successes, treatment_n, control_successes, control_n):
    """Run a two-sided pooled two-proportion z-test."""
    treatment_rate = treatment_successes / treatment_n
    control_rate = control_successes / control_n
    pooled_rate = (treatment_successes + control_successes) / (treatment_n + control_n)
    pooled_standard_error = sqrt(pooled_rate * (1 - pooled_rate) * (1 / treatment_n + 1 / control_n))
    z_statistic = (treatment_rate - control_rate) / pooled_standard_error
    p_value = two_sided_normal_p_value(z_statistic)
    return z_statistic, p_value, pooled_standard_error

In [6]:
control = customers.loc[customers["price_increase_occurred"] == False].copy()
treatment = customers.loc[customers["price_increase_occurred"] == True].copy()

control_n = len(control)
treatment_n = len(treatment)
control_churned = int(control["churned"].sum())
treatment_churned = int(treatment["churned"].sum())
control_churn_rate = control_churned / control_n
treatment_churn_rate = treatment_churned / treatment_n

absolute_difference = treatment_churn_rate - control_churn_rate
relative_difference = absolute_difference / control_churn_rate
ci_lower, ci_upper, unpooled_standard_error = difference_in_proportions_ci(
    treatment_churn_rate,
    control_churn_rate,
    treatment_n,
    control_n,
)
z_statistic, p_value, pooled_standard_error = two_proportion_z_test(
    treatment_churned,
    treatment_n,
    control_churned,
    control_n,
)

test_results = pd.DataFrame(
    {
        "metric": [
            "control sample size",
            "treatment sample size",
            "control churned customers",
            "treatment churned customers",
            "control churn rate",
            "treatment churn rate",
            "absolute churn-rate difference",
            "relative churn-rate difference",
            "95% CI lower bound",
            "95% CI upper bound",
            "z-statistic",
            "p-value",
        ],
        "value": [
            control_n,
            treatment_n,
            control_churned,
            treatment_churned,
            control_churn_rate,
            treatment_churn_rate,
            absolute_difference,
            relative_difference,
            ci_lower,
            ci_upper,
            z_statistic,
            p_value,
        ],
    }
)

display(test_results)

,metric,value
0,control sample size,"8,929.0000"
1,treatment sample size,"3,571.0000"
2,control churned customers,595.0000
3,treatment churned customers,315.0000
4,control churn rate,0.0666
5,treatment churn rate,0.0882
6,absolute churn-rate difference,0.0216
7,relative churn-rate difference,0.3238
8,95% CI lower bound,0.0109
9,95% CI upper bound,0.0322


## Practical Business Significance

Statistical significance asks whether the observed difference is unlikely under a no-difference hypothesis. Practical significance asks whether the size of the difference is large enough to matter for the business.

In [7]:
alpha = 0.05
practical_threshold = 0.01
statistically_significant = p_value < alpha
practically_significant = abs(absolute_difference) >= practical_threshold
incremental_churned_customers = absolute_difference * treatment_n

business_significance = pd.DataFrame(
    {
        "question": [
            "Is the difference statistically significant at alpha=0.05?",
            "Does the confidence interval exclude zero?",
            "Does the absolute difference meet the 1 percentage-point business threshold?",
            "Estimated incremental churned customers in treatment group",
        ],
        "answer": [
            statistically_significant,
            not (ci_lower <= 0 <= ci_upper),
            practically_significant,
            round(incremental_churned_customers, 1),
        ],
    }
)

display(business_significance)

,question,answer
0,Is the difference statistically significant at...,True
1,Does the confidence interval exclude zero?,True
2,Does the absolute difference meet the 1 percen...,True
3,Estimated incremental churned customers in tre...,77.0000


## A/B Test Findings

In [8]:
if statistically_significant:
    statistical_text = "statistically significant"
else:
    statistical_text = "not statistically significant"

if practically_significant:
    practical_text = "meets the practical threshold of 1 percentage point"
else:
    practical_text = "does not meet the practical threshold of 1 percentage point"

if absolute_difference > 0:
    direction_text = "higher"
elif absolute_difference < 0:
    direction_text = "lower"
else:
    direction_text = "the same"

print("A/B Test Findings")
print(f"- Control group size: {control_n:,} customers; churn rate: {control_churn_rate:.2%}.")
print(f"- Treatment group size: {treatment_n:,} customers; churn rate: {treatment_churn_rate:.2%}.")
print(
    f"- Treatment churn is {abs(absolute_difference) * 100:.2f} percentage points "
    f"{direction_text} than control, a relative difference of {relative_difference:.2%}."
)
print(
    f"- The 95% confidence interval for the treatment-control difference is "
    f"[{ci_lower * 100:.2f}, {ci_upper * 100:.2f}] percentage points."
)
print(f"- The two-proportion z-test gives z={z_statistic:.3f} and p={p_value:.4f}, so the observed difference is {statistical_text} at alpha=0.05.")
print(f"- The observed difference {practical_text}, with about {incremental_churned_customers:.1f} additional churned customers implied within the treatment group.")
print("- This result describes an experimental-style relationship between price exposure and churn in the synthetic data; it should not be interpreted as a causal effect until a separate causal design is specified.")

A/B Test Findings
- Control group size: 8,929 customers; churn rate: 6.66%.
- Treatment group size: 3,571 customers; churn rate: 8.82%.
- Treatment churn is 2.16 percentage points higher than control, a relative difference of 32.38%.
- The 95% confidence interval for the treatment-control difference is [1.09, 3.22] percentage points.
- The two-proportion z-test gives z=4.194 and p=0.0000, so the observed difference is statistically significant at alpha=0.05.
- The observed difference meets the practical threshold of 1 percentage point, with about 77.0 additional churned customers implied within the treatment group.
- This result describes an experimental-style relationship between price exposure and churn in the synthetic data; it should not be interpreted as a causal effect until a separate causal design is specified.
